## Quantization

### Dynamic Fixed-Point Quantization

In [37]:
import os
import pickle
import torch
from timesformer.models.vit import VisionTransformer
from functools import partial
import torch.nn as nn

torch.backends.quantized.engine = 'qnnpack'

with open(os.path.join("/Users/epheriami/Downloads/4B/Code/SYDE 673/P/tsformer/checkpoints/Exp3/args.pkl"), 'rb') as f:
    args = pickle.load(f)

f.close()
model_params = args["model_params"]

model = VisionTransformer(img_size=model_params["image_size"],
                          num_classes=model_params["num_classes"],
                          patch_size=model_params["patch_size"],
                          embed_dim=model_params["dim"],
                          depth=model_params["depth"],
                          num_heads=model_params["heads"],
                          mlp_ratio=4,
                          qkv_bias=True,
                          norm_layer=partial(nn.LayerNorm, eps=1e-6),
                          drop_rate=0.,
                          attn_drop_rate=0.,
                          drop_path_rate=0.1,
                          num_frames=model_params["num_frames"],
                          attention_type=model_params["attention_type"])

checkpoint = torch.load('/Users/epheriami/Downloads/4B/Code/SYDE 673/P/tsformer/checkpoints/Exp3/checkpoint_model3_exp20.pth', map_location=torch.device('cpu'))

model.load_state_dict(checkpoint['model_state_dict'])
model.eval()

quantized_model = torch.ao.quantization.quantize_dynamic(
    model, {torch.nn.Linear}, dtype=torch.qint8
)
quantized_model = quantized_model.to("cpu")
quantized_model.eval()

VisionTransformer(
  (dropout): Dropout(p=0.0, inplace=False)
  (patch_embed): PatchEmbed(
    (proj): Conv2d(3, 384, kernel_size=(16, 16), stride=(16, 16))
  )
  (pos_drop): Dropout(p=0.0, inplace=False)
  (time_drop): Dropout(p=0.0, inplace=False)
  (blocks): ModuleList(
    (0): Block(
      (norm1): LayerNorm((384,), eps=1e-06, elementwise_affine=True)
      (attn): Attention(
        (qkv): DynamicQuantizedLinear(in_features=384, out_features=1152, dtype=torch.qint8, qscheme=torch.per_tensor_affine)
        (proj): DynamicQuantizedLinear(in_features=384, out_features=384, dtype=torch.qint8, qscheme=torch.per_tensor_affine)
        (proj_drop): Dropout(p=0.0, inplace=False)
        (attn_drop): Dropout(p=0.0, inplace=False)
      )
      (temporal_norm1): LayerNorm((384,), eps=1e-06, elementwise_affine=True)
      (temporal_attn): Attention(
        (qkv): DynamicQuantizedLinear(in_features=384, out_features=1152, dtype=torch.qint8, qscheme=torch.per_tensor_affine)
        (proj): 

In [39]:
import numpy as np
from torch import nn
import os
from functools import partial
import pickle
import torch
from timesformer.models.vit import VisionTransformer

def params_count(model, ignore_bn=False):
    if not ignore_bn:
        return np.sum([p.numel() for p in model.parameters()]).item()
    else:
        count = 0
        for m in model.modules():
            if not isinstance(m, nn.BatchNorm3d):
                for p in m.parameters(recurse=False):
                    count += p.numel()
    return count


# print(f"Size (MB): {round(os.path.getsize(quantized_model)/1e6, 2)}")
print(f"Parameter count (M): {round(params_count(quantized_model)/1e6, 2)}")

Parameter count (M): 0.51


In [40]:
import sys

sys.getsizeof(quantized_model)

48

In [ ]:
quantized_model

VisionTransformer(
  (dropout): Dropout(p=0.0, inplace=False)
  (patch_embed): PatchEmbed(
    (proj): Conv2d(3, 384, kernel_size=(16, 16), stride=(16, 16))
  )
  (pos_drop): Dropout(p=0.0, inplace=False)
  (time_drop): Dropout(p=0.0, inplace=False)
  (blocks): ModuleList(
    (0): Block(
      (norm1): LayerNorm((384,), eps=1e-06, elementwise_affine=True)
      (attn): Attention(
        (qkv): DynamicQuantizedLinear(in_features=384, out_features=1152, dtype=torch.qint8, qscheme=torch.per_tensor_affine)
        (proj): DynamicQuantizedLinear(in_features=384, out_features=384, dtype=torch.qint8, qscheme=torch.per_tensor_affine)
        (proj_drop): Dropout(p=0.0, inplace=False)
        (attn_drop): Dropout(p=0.0, inplace=False)
      )
      (temporal_norm1): LayerNorm((384,), eps=1e-06, elementwise_affine=True)
      (temporal_attn): Attention(
        (qkv): DynamicQuantizedLinear(in_features=384, out_features=1152, dtype=torch.qint8, qscheme=torch.per_tensor_affine)
        (proj): 

### Post-Training Static Quantization

#### Approach 1

In [ ]:
import os
import pickle
import torch
from timesformer.models.vit import VisionTransformer
from functools import partial
import torch.nn as nn

torch.backends.quantized.engine = 'qnnpack'

# Load model and checkpoint
with open(os.path.join("/Users/epheriami/Downloads/4B/Code/SYDE 673/P/tsformer/checkpoints/Exp3/args.pkl"), 'rb') as f:
    args = pickle.load(f)

model_params = args["model_params"]

model = VisionTransformer(img_size=model_params["image_size"],
                          num_classes=model_params["num_classes"],
                          patch_size=model_params["patch_size"],
                          embed_dim=model_params["dim"],
                          depth=model_params["depth"],
                          num_heads=model_params["heads"],
                          mlp_ratio=4,
                          qkv_bias=True,
                          norm_layer=partial(nn.LayerNorm, eps=1e-6),
                          drop_rate=0.,
                          attn_drop_rate=0.,
                          drop_path_rate=0.1,
                          num_frames=model_params["num_frames"],
                          attention_type=model_params["attention_type"])

checkpoint = torch.load('/Users/epheriami/Downloads/4B/Code/SYDE 673/P/tsformer/checkpoints/Exp3/checkpoint_model3_exp20.pth', map_location=torch.device('cpu'))

model.load_state_dict(checkpoint['model_state_dict'])

model.eval()

# Define quantization configuration
model.qconfig = torch.ao.quantization.get_default_qconfig('qnnpack')

# Prepare model for static quantization (insert observers)
prepared_model = torch.ao.quantization.prepare(model)

# Calibrate with representative dataset (replace with your data loading logic)
def calibrate(model, calibration_data):
    model.eval()
    with torch.no_grad():
        for data in calibration_data:
            model(data)  # Use actual input data from your dataset

# Create representative input data (adapt to your input dimensions)
# This should be a representative sample from your dataset
# Correct calibration input creation with proper dimension order
num_calibration_samples = 100
calibration_input = torch.randn((
    num_calibration_samples,
    3,  # Color channels (moved to 2nd position)
    model_params["num_frames"],  # Temporal dimension (4 frames)
    model_params["image_size"][0],  # Height (192)
    model_params["image_size"][1]  # Width (640)
))

# Updated calibration function to process batches correctly
def calibrate(model, calibration_data, batch_size=32):
    model.eval()
    with torch.no_grad():
        # Process in batches to maintain dimension order
        for i in range(0, calibration_data.size(0), batch_size):
            batch = calibration_data[i:i+batch_size]
            model(batch)  # Shape: (batch_size, 3, 4, 192, 640)

# Run calibration with proper batch processing
calibrate(prepared_model, calibration_input, batch_size=32)

# Convert to quantized model
quantized_model = torch.ao.quantization.convert(prepared_model)

quantized_model = quantized_model.to("cpu")
quantized_model.eval()

VisionTransformer(
  (dropout): QuantizedDropout(p=0.0, inplace=False)
  (patch_embed): PatchEmbed(
    (proj): QuantizedConv2d(3, 384, kernel_size=(16, 16), stride=(16, 16), scale=0.018306566402316093, zero_point=128)
  )
  (pos_drop): QuantizedDropout(p=0.0, inplace=False)
  (time_drop): QuantizedDropout(p=0.0, inplace=False)
  (blocks): ModuleList(
    (0): Block(
      (norm1): QuantizedLayerNorm((384,), eps=1e-06, elementwise_affine=True)
      (attn): Attention(
        (qkv): QuantizedLinear(in_features=384, out_features=1152, scale=0.033226776868104935, zero_point=129, qscheme=torch.per_tensor_affine)
        (proj): QuantizedLinear(in_features=384, out_features=384, scale=0.0007612815825268626, zero_point=120, qscheme=torch.per_tensor_affine)
        (proj_drop): QuantizedDropout(p=0.0, inplace=False)
        (attn_drop): QuantizedDropout(p=0.0, inplace=False)
      )
      (temporal_norm1): QuantizedLayerNorm((384,), eps=1e-06, elementwise_affine=True)
      (temporal_attn): 

#### Approach 2

In [36]:
import os
import pickle
import torch
from timesformer.models.vit import VisionTransformer
from functools import partial
import torch.nn as nn

# Load model and checkpoint
with open(os.path.join("/Users/epheriami/Downloads/4B/Code/SYDE 673/P/tsformer/checkpoints/Exp3/args.pkl"), 'rb') as f:
    args = pickle.load(f)

model_params = args["model_params"]

model = VisionTransformer(img_size=model_params["image_size"],
                          num_classes=model_params["num_classes"],
                          patch_size=model_params["patch_size"],
                          embed_dim=model_params["dim"],
                          depth=model_params["depth"],
                          num_heads=model_params["heads"],
                          mlp_ratio=4,
                          qkv_bias=True,
                          norm_layer=partial(nn.LayerNorm, eps=1e-6),
                          drop_rate=0.,
                          attn_drop_rate=0.,
                          drop_path_rate=0.1,
                          num_frames=model_params["num_frames"],
                          attention_type=model_params["attention_type"])

checkpoint = torch.load('/Users/epheriami/Downloads/4B/Code/SYDE 673/P/tsformer/checkpoints/Exp3/checkpoint_model3_exp20.pth', map_location=torch.device('cpu'))
model.load_state_dict(checkpoint['model_state_dict'])
model.eval()

# ======================
# Mac-compatible FP16 Conversion
# ======================
# Directly convert model parameters to float16
model = model.to(torch.float64)

# Create sample input in float16
num_calibration_samples = 100
calibration_input = torch.randn((
    num_calibration_samples,
    3,  # Color channels
    model_params["num_frames"],
    model_params["image_size"][0],
    model_params["image_size"][1]
), dtype=torch.float64)

# Verify forward pass
with torch.no_grad():
    output = model(calibration_input)
    print("Output shape:", output.shape)
    print("Output dtype:", output.dtype)  # Should be torch.float16

# Save quantized model
quantized_model = model.to("cpu")
quantized_model.eval()

KeyboardInterrupt: 

In [34]:
# Create state dictionary for quantized model
quantized_state = {
    "model_state_dict": quantized_model.state_dict(),
    "optimizer_state_dict": None,  # Quantized models typically don't need optimizer states
}

# Save quantized model checkpoint
torch.save(quantized_state, "static_quant_float16.pth")

In [32]:
from datasets.kitti import KITTI
from torchvision import transforms
from tqdm import tqdm
import numpy as np

sequences = ["01", "03", "04", "05", "06", "07", "10"]
device = "cpu"

preprocess = transforms.Compose([
    transforms.Resize(model_params["image_size"]),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.34721234, 0.36705238, 0.36066107],
        std=[0.30737526, 0.31515116, 0.32020183]),
    transforms.ConvertImageDtype(torch.float16),
])

for sequence in sequences:
    dataset = KITTI(transform=preprocess,
                   sequences=[sequence],
                   window_size=args["window_size"],
                   overlap=args["overlap"])
    test_loader = torch.utils.data.DataLoader(dataset, batch_size=1, shuffle=False)

    with tqdm(test_loader, unit="batch") as batchs:
        # Initialize with empty tensor list
        pred_poses = []
        batchs.set_description(f"Sequence {sequence}")
        
        for images, gt in batchs:
            with torch.no_grad():
                # Convert inputs and model
                images = images.to(device).half()
                quantized_model = quantized_model.to(device).half()
                
                # Get prediction
                pred_pose = quantized_model(images)
                
                # Handle dimensional mismatch
                if pred_pose.dim() == 2:
                    pred_pose = pred_pose.unsqueeze(1)  # Add temporal dimension
                
                # Store predictions
                pred_poses.append(pred_pose.cpu())

        # Concatenate all predictions at once
        if pred_poses:
            pred_poses = torch.cat(pred_poses, dim=0)
            print(f"Final predictions shape: {pred_poses.shape}")

Sequence 01:   6%|▋         | 23/366 [00:59<14:50,  2.60s/batch]


KeyboardInterrupt: 

In [ ]:
# Default weight size

checkpoint['model_state_dict']

for key, value in checkpoint['model_state_dict'].items():
    print(f"Key: {key}, Datatype: {value.dtype}")
    break

Key: cls_token, Datatype: torch.float32
